# 05 · Segment Classifier with Random Forest

We now use the segmentation found with K-Means as the target label to train a **Random Forest** classifier, so that new customers can be assigned to a segment directly. The final model is saved to `models/segmentation_model.pkl`.

## 1. Load the segmented data

In [1]:
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

SEGMENTED_PARQUET = "../data/processed/segmented_customers.parquet"
MODEL_PATH = "../models/segmentation_model.pkl"

DATA = pd.read_parquet(SEGMENTED_PARQUET)
DATA.columns

Index(['customer_id', 'age', 'gender', 'country', 'channel', 'monthly_spend',
       'clicks', 'impressions', 'conversions', 'signup_date', 'segment',
       'segment_name'],
      dtype='object')

## 2. Baseline model — all features

Categorical columns are one-hot encoded.

In [2]:
FEATURES = ['age', 'gender', 'country', 'channel', 'monthly_spend', 'clicks', 'impressions', 'conversions']
X = DATA[FEATURES]
X = pd.get_dummies(X, columns=['gender', 'country', 'channel'])
y = DATA['segment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
report_baseline = classification_report(y_test, y_pred)
print(report_baseline)

              precision    recall  f1-score   support

           0       0.82      1.00      0.90        23
           1       0.86      0.92      0.89        26
           2       0.87      0.74      0.80        27
           3       0.95      0.83      0.89        24

    accuracy                           0.87       100
   macro avg       0.88      0.87      0.87       100
weighted avg       0.88      0.87      0.87       100



## 3. Feature importance

In [3]:
importances = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(importances.round(3))

conversions          0.287
impressions          0.248
clicks               0.229
monthly_spend        0.112
age                  0.050
channel_Google       0.010
gender_Male          0.007
country_Chile        0.006
country_Colombia     0.006
channel_Facebook     0.006
country_Mexico       0.005
channel_TikTok       0.005
channel_Instagram    0.005
gender_Female        0.005
country_Peru         0.005
gender_Other         0.005
country_Argentina    0.004
channel_Email        0.004
dtype: float64


## 4. Refined model — top behavioural features only

Demographic and channel features contribute very little, so we keep only the four most important features.

In [4]:
FEATURES = ['conversions', 'impressions', 'clicks', 'monthly_spend']
X = DATA[FEATURES]
y = DATA['segment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
refined_model = RandomForestClassifier(n_estimators=100, random_state=0, max_depth=5)
refined_model.fit(X_train, y_train)
y_pred = refined_model.predict(X_test)
report_refined = classification_report(y_test, y_pred)
print(report_refined)

              precision    recall  f1-score   support

           0       0.88      1.00      0.94        23
           1       0.90      1.00      0.95        26
           2       0.91      0.78      0.84        27
           3       0.95      0.88      0.91        24

    accuracy                           0.91       100
   macro avg       0.91      0.91      0.91       100
weighted avg       0.91      0.91      0.91       100



## 5. Comparison between both models

In [5]:
print("=== Refined model (4 features) ===")
print(report_refined)
print("=== Baseline model (all features) ===")
print(report_baseline)

=== Refined model (4 features) ===
              precision    recall  f1-score   support

           0       0.88      1.00      0.94        23
           1       0.90      1.00      0.95        26
           2       0.91      0.78      0.84        27
           3       0.95      0.88      0.91        24

    accuracy                           0.91       100
   macro avg       0.91      0.91      0.91       100
weighted avg       0.91      0.91      0.91       100

=== Baseline model (all features) ===
              precision    recall  f1-score   support

           0       0.82      1.00      0.90        23
           1       0.86      0.92      0.89        26
           2       0.87      0.74      0.80        27
           3       0.95      0.83      0.89        24

    accuracy                           0.87       100
   macro avg       0.88      0.87      0.87       100
weighted avg       0.88      0.87      0.87       100



### Conclusions

Keeping only the four behavioural features improves accuracy from **87 %** to **91 %**: the demographic and channel dummies were adding noise rather than signal.

The refined model performs well overall, and the F1-scores confirm it is not just accurate but also balanced between precision and recall across all segments. It is most precise when identifying **Core Buyers** (precision 0.95): when it predicts that a customer belongs to that segment, it is right 95 % of the time. **Ghosts** have a perfect recall of 1.00, meaning the model never misses a real Ghost. This is valuable because, in a marketing context, you don't want to waste budget on someone who won't engage.

## 6. Save the model

In [6]:
joblib.dump(refined_model, MODEL_PATH)

['../models/segmentation_model.pkl']